Consider a system of $N$ populations, whose state and external input are described by $N$-dimensional vectors $\bar{r}$ and $\bar{h}$, respectively.

A population recieves recurrent inputs from all populations (including itself) and an external input. \
Let's denote the input-output relation for the $n$-th population as $f_n$:
$$
r_n^{out} = f_n(\bar{r}^{in}, h_n)
$$

In vectorized form:
$$
\bar{r}^{out} = F(\bar{r}^{in}, \bar{h}), \\
F = (f_1, ... , f_n)
$$

Let's implement $F$ for a simple Wilson-Cowan-like model:
$$
\bar{r}^{out} = g(W \bar{r}^{in} + \bar{h}),
$$
where $W$ is an $N \times N$ weight matrix, and $g$ is a gain function (applied element-wise):
$$
g(x) = A / [1 + \exp(-k(x - x_c))]
$$

In [ ]:
import numpy as np
import pandas as pd


# Gain function
def g(x, gpar):
    return gpar['A'] / (1 + np.exp(-gpar['k'] * (x - gpar['xc'])))

# F: (r_in, h) -> r_out (vectorized)
def F(r_in, h, W, gpar) -> np.ndarray:
    return g(W @ r_in + h, gpar)

# fpop: (r_in, h) -> r_out (single population)
def fpop(r_in, h, pop_num, W, gpar) -> float:
    return g(W[pop_num, :] @ r_in + h[pop_num], gpar).item()

# Set the parameters
N = 5  # Number of populations
gpar = {'A': 10, 'k': 0.5, 'xc': 0} # Gain parameters
W = 0.2 * np.random.randn(N, N)  # Weight matrix

# Inputs
r_in = 10 * np.abs(np.random.rand(N, 1))  # Input pop. rates
h = 1 * np.random.randn(N, 1)  # Exernal inputs

# Output firing rates (vectorized)
r_out = F(r_in, h, W, gpar)

# Print the result
df = pd.DataFrame(
    {'r_in': r_in.ravel(), 'h': h.ravel(), 'r_out': r_out.ravel()}
)
print(df.round(2))

# Test fpop
pop_num = 2
r_out_single = fpop(r_in, h, pop_num, W, gpar)
print(f'\nfpop: {r_out_single:.2f}')
print(f'F: {r_out[pop_num].item():.2f}')


   r_in     h  r_out
0  8.44 -0.89   2.99
1  2.25  0.10   6.86
2  5.94  0.13   3.88
3  2.07  0.54   7.99
4  2.15  1.36   6.72

fpop: 3.88
F: 3.88


In a steady state, $\bar{r}$ should satisfy the self-consistency condition:
$$
\bar{r} = F(\bar{r}, \bar{h})
$$

Let's define a function $S: \bar{h} \rightarrow \bar{r}$ that **maps an external input $\bar{h}$ to the corresponding solution** $\bar{r}$ of the  above equation. In other words:
$$
S(\bar{h}) = F(S(\bar{h}), \bar{h})
$$

To implement $S$, we can take an initial guess $\bar{r}=\bar{r}_{start}$ and recursively apply $F$ to it several times. For better convergence, we can introduce a factor $\alpha$ that defines how much the curent guess changes at each iteration.

In [49]:
def S(r_start, h, W, gpar, n_iter=10, alpha=0.5):
    r = r_start
    for n in range(n_iter):
        rnew = F(r, h, W, gpar)
        r = alpha * rnew + (1 - alpha) * r
    return r

Let's choose an external input $\bar{h}_0$ and calculate the steady state using the above implementation of $S$:
$$
\bar{r}_0=S(\bar{h}_0)) 
$$
If it works correctly, then:
$$
F(\bar{r}_0, \bar{h}_0) \approx \bar{r}_0
$$.
Let's check it.

In [52]:
# External input
h0 = 1 * np.random.randn(N, 1)

# First guess
r_start = np.zeros((N, 1))

# Steady state
r0 = S(r_start, h0, W, gpar, n_iter=10, alpha=0.8)

# Check the result
print('r0:')
print(r0.ravel())
print('F(r0):')
print(F(r0, h0, W, gpar).ravel())

r0:
[1.84556026 8.14871877 3.01522158 4.75355903 5.18409443]
F(r0):
[1.84556009 8.1487958  3.01520859 4.75360008 5.18414685]


Now the question is: **how a small perturbation of the external input will affect the steady state?** \
In other words:
$$
S(\bar{h}_0 + \Delta\bar{h}) = ?
$$

Let's write the perturbed state in the form: $\bar{r}_0 + \Delta\bar{r}$. It should satisfy:
$$
\bar{r}_0 + \Delta\bar{r} = F(\bar{r}_0 + \Delta\bar{r}, \bar{h}_0 + \Delta\bar{h})
$$

Let's expand $F$ about $(\bar{r}_0, \bar{h}_0)$ up to the 1-st order:
$$ 
\bar{r}_0 + \Delta\bar{r} \approx F(\bar{r}_0, \bar{h}_0) + J \Delta\bar{r} + Q \Delta\bar{h}
$$
where $J$ and $Q$ are two matrices, given by:
$$
J_{kn} = {\partial f_k} / {\partial r_n} |_{r_0, h_0} \\
Q_{nn} = {\partial f_n} / {\partial h_n} |_{r_0, h_0} \\
Q_{kn} = 0 \text{  for  } k \neq n
$$

Because $\bar{r}_0 = F(\bar{r}_0, \bar{h}_0)$, we get a linear system:
$$
\Delta\bar{r} = J \Delta\bar{r} + Q \Delta\bar{h}
$$

Its solution is:
$$
\Delta\bar{r} = (E - J)^{-1} Q \Delta\bar{h}
$$
whre $E$ is unity matrix.

Thus, **the perturbed steady state can be approximated** as:
$$
S(\bar{h}_0 + \Delta\bar{h}) \approx S(\bar{h}_0) + (E - J)^{-1} Q \Delta\bar{h}
$$

Now we should calculate the matrices $J$ and $Q$. In our example, they can be found explicitly, because $F$ is a known function that is easy to differentiate. But in general case, the derivatives should be found **numerically**.

Let's choose a small scalar value $\epsilon_r$ and define a vector $\Delta\bar{r}_n$, whose length is $\epsilon_r$ and the direction is parallel to the $n$-th axis. In other words, the $n$-th component of $\Delta\bar{r}_n$ is $\epsilon_r$, and all other components equal to zero:
$$
\Delta\bar{r}_n = (0, ..., \epsilon_r, ..., 0)^T
$$

Now $J_{kn}$ can be approximated as:
$$
J_{kn} \approx [f_k(\bar{r}_0 + \Delta\bar{r}_n, \bar{h}_0) - f_k(\bar{r}_0, \bar{h}_0)] / \epsilon_r
$$

Similarly, we can choose $\epsilon_h$, define $\Delta\bar{h}_n$, and estimate $Q_{kn}$.

In [63]:
# Population index
k = 1
# Axis to perturb
n = 2

# Perturbation size
eps_r = 0.1

# Vector of rate perturbations along the n-th axis
dr_n = np.zeros((N, 1))
dr_n[n] = eps_r

# Jkn = dfk / drn
df = (
    fpop(r0 + dr_n, h0, k, W, gpar) - fpop(r0, h0, k, W, gpar)
)
Jkn = df / eps_r

print(f'J_kn = {Jkn:.4f}')

J_kn = 0.0523


**Task description**:

- Compute all the elements of the matrices $J$ and $Q$

- Choose an arbitrary vector of external input perturbation $\Delta\bar{h}$. All its components shoould be non-zero.

- Predict the effect of $\Delta\bar{h}$ on the system's state: $\hat{\bar{r}} = r_0 + (E - J)^{-1} Q \Delta\bar{h}$

- Compare it with the actual effect: $\bar{r} = S(\bar{h}_0 + \Delta\bar{h})$

- Explore how the error $|\hat{\bar{r}} - \bar{r}|$ depends on $|\Delta\bar{h}|$
